# Finite-State Morphology: Rules That Run Both Ways

**Computational Text Analysis Pack · Module 14 — Preparing Text, Properly**

*Technique:* Two-level morphology; a reversible analyser/generator for Swahili verbs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/14b_finite_state_morphology.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Everything the course has done so far treats a word as an atom. For English that
is a tolerable simplification. For Swahili, Hausa, Yorùbá or Zulu it throws away
most of the grammar, because those languages carry in a single word what English
spreads across a clause.

**Ninasoma** is one Swahili token and four pieces of information: *ni-* (I),
*-na-* (present), *soma* (read). A word-frequency list sees a unique word. A
morphological analyser sees a first-person present-tense form of a verb it
already knows.

This is the classic symbolic method, and the survey is explicit about why it
still matters: a finite-state transducer is **reversible** — the same machine
analyses and generates — and **auditable**, because every output traces to a
rule you wrote.

## The tooling

The industrial versions of this are **Foma** (open source), **HFST** (Helsinki),
and **Pynini** (Google, built on OpenFst); the classic reference is Beesley and
Karttunen's *Finite-State Morphology*, which documents Xerox's `xfst`/`lexc`.
None are installed here, and none are needed to see the idea: a transducer over
a small alphabet is a few dozen lines of plain Python.

Where this beats a statistical tagger: small data, morphologically rich
languages, and any case where you must be able to explain why the analyser said
what it said.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# A two-level lexicon: each level is a set of morphemes with a gloss.
SUBJECT = {"ni": "1SG", "u": "2SG", "a": "3SG", "tu": "1PL",
           "m": "2PL", "wa": "3PL"}
TENSE = {"na": "PRES", "li": "PAST", "ta": "FUT", "me": "PERF", "ki": "COND"}
NEGATION = {"ha": "NEG"}
ROOTS = {"soma": "read", "andika": "write", "penda": "love", "ona": "see",
         "sema": "say", "fanya": "do", "enda": "go", "taka": "want",
         "jua": "know", "pata": "get", "kuja": "come", "lala": "sleep"}

print(f"{len(SUBJECT)} subject markers x {len(TENSE)} tenses x {len(ROOTS)} roots")
print(f"= {len(SUBJECT) * len(TENSE) * len(ROOTS)} verb forms from "
      f"{len(SUBJECT) + len(TENSE) + len(ROOTS)} stored pieces.")
print()
print("That ratio is the argument for morphology. A word list would need to")
print("store every form; a transducer stores the parts and the rules.")

## Direction one: analysis

Walk the string left to right, consuming the longest morpheme that matches at
each position. If the whole string is consumed and a root was found, the parse
succeeds.

In [ ]:
def analyse(word):
    """Surface form -> list of (morpheme, gloss). Empty list if unanalysable."""
    pos, out = 0, []
    neg = False
    for surface, gloss in sorted(NEGATION.items(), key=lambda kv: -len(kv[0])):
        if word.startswith(surface):
            out.append((surface, gloss))
            pos += len(surface)
            neg = True
            break
    for table, label in [(SUBJECT, "subject"), (TENSE, "tense")]:
        for surface, gloss in sorted(table.items(), key=lambda kv: -len(kv[0])):
            if word.startswith(surface, pos):
                out.append((surface, gloss))
                pos += len(surface)
                break
        else:
            if label == "subject":
                return []
    rest = word[pos:]
    if rest in ROOTS:
        out.append((rest, ROOTS[rest]))
        return out
    return []

for w in ["ninasoma", "tulisoma", "watataka", "hawapendi", "amelala", "nyumba"]:
    parse = analyse(w)
    if parse:
        print(f"  {w:<12} = " + " + ".join(f"{m}({g})" for m, g in parse))
    else:
        print(f"  {w:<12} = (not analysable by this grammar)")

## Direction two: generation

This is the property that distinguishes a transducer from a parser. The same
rules, read the other way, *produce* forms. If analysis and generation do not
round-trip, the grammar is inconsistent — and you find out immediately.

In [ ]:
def generate(subject, tense, root):
    inv_s = {v: k for k, v in SUBJECT.items()}
    inv_t = {v: k for k, v in TENSE.items()}
    inv_r = {v: k for k, v in ROOTS.items()}
    return inv_s[subject] + inv_t[tense] + inv_r[root]

print("generation:")
for spec in [("1SG", "PRES", "read"), ("3PL", "FUT", "want"),
             ("1PL", "PAST", "see"), ("2SG", "PERF", "come")]:
    print(f"  {spec} -> {generate(*spec)}")

print("\nround-trip check (generate, then analyse, and compare):")
ok = 0
total = 0
for s in SUBJECT.values():
    for t in TENSE.values():
        for r in ROOTS.values():
            total += 1
            form = generate(s, t, r)
            back = analyse(form)
            if back and [g for _, g in back] == [s, t, r]:
                ok += 1
print(f"  {ok}/{total} forms round-trip exactly")
print("\nA statistical tagger cannot make this guarantee. This one can, because")
print("the rules ARE the model.")

## Against real text

A toy grammar with twelve roots will not analyse a news corpus. The useful
question is how much it *does* reach, and what the failures look like — that is
how a real morphological grammar gets built, one failure class at a time.

In [ ]:
swahili = df[df["language"] == "Swahili"]
toks = Counter()
for t in swahili["text"]:
    toks.update(tokenise(t))
print(f"{sum(toks.values()):,} Swahili tokens, {len(toks):,} distinct")

analysed = {w: analyse(w) for w in toks if analyse(w)}
covered = sum(toks[w] for w in analysed)
print(f"\nthis 12-root grammar analyses {len(analysed)} distinct forms, "
      f"{covered:,} tokens ({covered / sum(toks.values()):.2%})")
print()
for w in sorted(analysed, key=lambda w: -toks[w])[:10]:
    print(f"  {w:<14}{toks[w]:>5}  " +
          " + ".join(f"{m}({g})" for m, g in analysed[w]))
print()
print("Low coverage is the expected result and the honest one. Real analysers")
print("for Swahili have thousands of roots and handle noun classes, object")
print("infixes and relative markers. The method scales; this lexicon does not.")

### 🔧 Your turn 1

Add three roots you know to `ROOTS` and re-run the coverage cell. How much does each new root buy you?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Each root adds up to thirty forms at once, because it combines with every
subject and tense — which is the compounding return that makes hand-built
morphology worth the labour for a language with good descriptive grammars.
Coverage still climbs slowly, because token frequency in news text is dominated
by nouns and borrowings this grammar does not touch at all. The realistic path
is a published lexicon plus rules, which is exactly what HFST and Foma
grammars for African languages are: years of linguistic description, compiled.

</details>